# Gradient Descent Variants

This notebook accompanies the **ML Viz** lesson on Gradient Descent Variants.

We implement SGD, Momentum, RMSprop, and Adam from scratch, visualize their
trajectories on the 2D Rosenbrock function, inspect Adam's bias-correction step
by step, and plot common learning rate schedules.

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/optimization-ml/01-gradient-descent-variants

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#2e3347',
    'axes.labelcolor':  '#94a3b8',
    'text.color':       'white',
    'xtick.color':      '#94a3b8',
    'ytick.color':      '#94a3b8',
    'grid.color':       '#2e3347',
    'grid.alpha':       0.4,
    'legend.facecolor': '#1a1d27',
    'legend.edgecolor': '#2e3347',
})

rng = np.random.default_rng(42)

# Brand colour palette
C_BRAND  = '#818cf8'  # indigo  — SGD
C_TEAL   = '#14b8a6'  # teal    — Momentum
C_YELLOW = '#f59e0b'  # yellow  — RMSprop
C_ROSE   = '#f43f5e'  # rose    — Adam

## 1  Optimizer trajectories on the Rosenbrock function

The **Rosenbrock banana** function is the standard stress-test for optimizers:

$$f(x, y) = (1 - x)^2 + 100\,(y - x^2)^2$$

Its global minimum is at $(x, y) = (1, 1)$ inside a narrow curved valley.
Plain SGD zig-zags across the steep valley walls while barely advancing along
the floor; momentum and adaptive methods handle it far better.

We run each optimizer for **200 steps** from the start point $(-1, 1)$
with $\eta = 0.001$.

In [ ]:
# ── Rosenbrock function and gradient ──────────────────────────────────────────
def rosenbrock(x, y):
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_grad(x, y):
    dx = -2 * (1 - x) - 400 * x * (y - x**2)
    dy = 200 * (y - x**2)
    return np.array([dx, dy])

# ── Optimizer implementations ─────────────────────────────────────────────────
def run_sgd(start, steps=200, lr=0.001):
    p = np.array(start, dtype=float)
    path = [p.copy()]
    for _ in range(steps):
        g = rosenbrock_grad(*p)
        p -= lr * g
        path.append(p.copy())
    return np.array(path)

def run_momentum(start, steps=200, lr=0.001, beta=0.9):
    p = np.array(start, dtype=float)
    v = np.zeros(2)
    path = [p.copy()]
    for _ in range(steps):
        g = rosenbrock_grad(*p)
        v = beta * v + (1 - beta) * g
        p -= lr * v
        path.append(p.copy())
    return np.array(path)

def run_rmsprop(start, steps=200, lr=0.001, rho=0.9, eps=1e-8):
    p = np.array(start, dtype=float)
    G = np.zeros(2)
    path = [p.copy()]
    for _ in range(steps):
        g = rosenbrock_grad(*p)
        G = rho * G + (1 - rho) * g**2
        p -= lr * g / (np.sqrt(G) + eps)
        path.append(p.copy())
    return np.array(path)

def run_adam(start, steps=200, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    p = np.array(start, dtype=float)
    m = np.zeros(2)
    v = np.zeros(2)
    path = [p.copy()]
    for t in range(1, steps + 1):
        g = rosenbrock_grad(*p)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g**2
        m_hat = m / (1 - b1**t)
        v_hat = v / (1 - b2**t)
        p -= lr * m_hat / (np.sqrt(v_hat) + eps)
        path.append(p.copy())
    return np.array(path)

# ── Run all optimizers ────────────────────────────────────────────────────────
START = [-1.0, 1.0]
paths = {
    'SGD':      run_sgd(START),
    'Momentum': run_momentum(START),
    'RMSprop':  run_rmsprop(START),
    'Adam':     run_adam(START),
}
colors = {
    'SGD': C_BRAND, 'Momentum': C_TEAL, 'RMSprop': C_YELLOW, 'Adam': C_ROSE
}

print("Final losses after 200 steps:")
for name, path in paths.items():
    x, y = path[-1]
    print(f"  {name:10s}  f={rosenbrock(x, y):.4f}  pos=({x:.4f}, {y:.4f})")

# ── Contour background ────────────────────────────────────────────────────────
xs = np.linspace(-2, 2, 400)
ys = np.linspace(-0.5, 3, 400)
XX, YY = np.meshgrid(xs, ys)
ZZ = rosenbrock(XX, YY)

fig, (ax_traj, ax_loss) = plt.subplots(1, 2, figsize=(14, 5.5))

# Trajectory panel
ax_traj.contourf(XX, YY, np.log1p(ZZ), levels=40, cmap='inferno', alpha=0.55)
ax_traj.contour( XX, YY, np.log1p(ZZ), levels=40, colors='white', alpha=0.08, linewidths=0.4)
for name, path in paths.items():
    ax_traj.plot(path[:, 0], path[:, 1], '-', color=colors[name],
                 linewidth=1.6, alpha=0.85, label=name)
    ax_traj.plot(*path[0],  'o', color=colors[name], markersize=5)
    ax_traj.plot(*path[-1], 's', color=colors[name], markersize=5)
ax_traj.scatter([1], [1], s=140, color='white', zorder=10, marker='*', label='Minimum (1,1)')
ax_traj.scatter(*START, s=80, color='#94a3b8', zorder=10, label='Start (-1,1)')
ax_traj.set_xlim(-2, 2); ax_traj.set_ylim(-0.5, 3)
ax_traj.set_xlabel('x'); ax_traj.set_ylabel('y')
ax_traj.set_title('Rosenbrock trajectories (200 steps, lr=0.001)', color='white')
ax_traj.legend(fontsize=9)

# Loss curves panel
for name, path in paths.items():
    losses = [rosenbrock(*pt) for pt in path]
    ax_loss.semilogy(losses, color=colors[name], linewidth=1.8, label=f"{name} final={losses[-1]:.3f}")
ax_loss.set_xlabel('Step'); ax_loss.set_ylabel('f(x, y) — log scale')
ax_loss.set_title('Convergence on Rosenbrock', color='white')
ax_loss.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 2  Adam step-by-step: bias correction in action

Adam initialises both moment estimates at zero:
$m_0 = v_0 = 0$. At $t=1$ with $\beta_1 = 0.9$ and true gradient $g$:

$$m_1 = (1-\beta_1)\,g = 0.1\,g \quad\Rightarrow\quad m_1 \text{ is 10\texttimes{} too small!}$$

Bias correction divides by $(1-\beta^t)$ to recover the true scale:

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \qquad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

The cell below runs 5 Adam steps on the scalar loss $L(\theta) = \theta^2$
(gradient $= 2\theta$) and prints $\theta$, $m$, $v$, $\hat{m}$, $\hat{v}$
at every step — watch how $\hat{m}$ and $\hat{v}$ stay close to the true
gradient ($\approx 2$) and squared gradient ($\approx 4$) even at $t=1$.

In [ ]:
def adam_step(theta, grad, m, v, t, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    """One Adam update. Returns (new_theta, new_m, new_v)."""
    m = b1 * m + (1 - b1) * grad
    v = b2 * v + (1 - b2) * grad**2
    m_hat = m / (1 - b1**t)
    v_hat = v / (1 - b2**t)
    theta = theta - lr * m_hat / (np.sqrt(v_hat) + eps)
    return theta, m, v

print(f"{'t':>3}  {'theta':>10}  {'m (raw)':>12}  {'v (raw)':>12}  "
      f"{'m_hat':>10}  {'v_hat':>10}")
print("-" * 65)

theta, m, v = 1.0, 0.0, 0.0
for t in range(1, 6):
    g = 2.0 * theta  # gradient of theta^2
    theta, m, v = adam_step(theta, g, m, v, t)
    m_hat = m / (1 - 0.9**t)
    v_hat = v / (1 - 0.999**t)
    print(f"{t:>3}  {theta:>10.6f}  {m:>12.6f}  {v:>12.8f}  "
          f"{m_hat:>10.4f}  {v_hat:>10.4f}")

print()
print("Note: m_hat and v_hat stay near the true gradient (~2) and")
print("squared gradient (~4) even at t=1, thanks to bias correction.")

## 3  Learning rate schedules

A fixed learning rate is rarely optimal. Three common schedules:

| Schedule | Formula |
|----------|---------|
| Constant | $\eta_t = \eta_0$ |
| Step decay | $\eta_t = \eta_0 \cdot \gamma^{\lfloor t/k \rfloor}$ |
| Cosine annealing | $\eta_t = \eta_{\min} + \tfrac{1}{2}(\eta_{\max} - \eta_{\min})\bigl(1 + \cos(\pi t / T)\bigr)$ |

Cosine annealing slows naturally near the minimum, giving the optimizer more
time to fine-tune. Warm restarts (SGDR) periodically reset $t \leftarrow 0$
to escape local minima and explore new regions.

In [ ]:
T = 100  # total epochs
epochs = np.arange(T + 1)

lr_max, lr_min = 0.1, 0.0

def constant_lr(t, T, lr_max, lr_min=0.0):
    return np.full_like(t, lr_max, dtype=float)

def step_decay(t, T, lr_max, lr_min=0.0, gamma=0.5, step=20):
    return lr_max * (gamma ** (t // step))

def cosine_lr_schedule(t, T, lr_max, lr_min=0.0):
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * t / T))

sched_constant = constant_lr(epochs, T, lr_max)
sched_step     = step_decay(epochs, T, lr_max)
sched_cosine   = cosine_lr_schedule(epochs, T, lr_max)

fig, ax = plt.subplots(figsize=(10, 4.5))

ax.plot(epochs, sched_constant, color='#64748b', linewidth=2.0,
        linestyle='--', label='Constant  $\\eta=0.1$')
ax.plot(epochs, sched_step,     color=C_YELLOW, linewidth=2.0,
        label='Step decay  $\\gamma=0.5$, $k=20$')
ax.plot(epochs, sched_cosine,   color=C_TEAL,   linewidth=2.5,
        label='Cosine annealing')

# Annotate cosine formula
mid_t = T // 2
mid_lr = sched_cosine[mid_t]
ax.annotate(
    r'$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max}-\eta_{\min})(1+\cos(\pi t/T))$',
    xy=(mid_t, mid_lr),
    xytext=(mid_t - 28, mid_lr + 0.028),
    color=C_TEAL,
    fontsize=9.5,
    arrowprops=dict(arrowstyle='->', color=C_TEAL, lw=1.2),
)

# Mark endpoints of cosine
ax.scatter([0, T], [sched_cosine[0], sched_cosine[-1]],
           color=C_TEAL, s=60, zorder=10)
ax.text(2, sched_cosine[0] + 0.003, f'$\\eta_{{max}}={lr_max}$',
        color=C_TEAL, fontsize=9)
ax.text(T - 18, sched_cosine[-1] + 0.005, f'$\\eta_{{min}}={lr_min}$',
        color=C_TEAL, fontsize=9)

ax.set_xlabel('Epoch')
ax.set_ylabel('Learning rate $\\eta$')
ax.set_title('Learning rate schedules over 100 epochs', color='white')
ax.set_xlim(0, T)
ax.set_ylim(-0.005, 0.115)
ax.legend(fontsize=10)
ax.grid(True)
plt.tight_layout()
plt.show()

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: each concept is recapped, the code
outline is set, and `# TODO(you)` marks what you fill in. Run the `assert`
cell after each exercise — it passes silently (with a ✅ message) when correct.

### Exercise 1 — Implement one Adam step

Adam maintains two exponential moving averages of the gradient and its square,
then applies bias correction before updating the parameter:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\,g_t \qquad\text{(first moment)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)\,g_t^2 \qquad\text{(second moment)}$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t} \qquad\text{(bias correction)}$$
$$\theta_t = \theta_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t} + \varepsilon}\,\hat{m}_t \qquad\text{(update)}$$

Implement `adam_update` below. It takes the current parameter `theta`, the
gradient `grad`, the current moment estimates `m` and `v`, and the timestep `t`.

In [ ]:
def adam_update(theta, grad, m, v, t, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    """One Adam update step.

    Args:
        theta: current parameter value (scalar or array)
        grad:  gradient at theta
        m:     first moment estimate (same shape as theta)
        v:     second moment estimate (same shape as theta)
        t:     current timestep (1-indexed)

    Returns:
        (new_theta, new_m, new_v)
    """
    # TODO(you): update first moment: b1 * m + (1 - b1) * grad
    m = ...

    # TODO(you): update second moment: b2 * v + (1 - b2) * grad**2
    v = ...

    # TODO(you): bias-corrected first moment: m / (1 - b1**t)
    m_hat = ...

    # TODO(you): bias-corrected second moment: v / (1 - b2**t)
    v_hat = ...

    # TODO(you): parameter update: theta - lr * m_hat / (sqrt(v_hat) + eps)
    theta = ...

    return theta, m, v

In [ ]:
# Checks — run me after filling in adam_update above
theta0 = 1.0
m0, v0 = 0.0, 0.0

theta1, m1, v1 = adam_update(theta0, 1.0, m0, v0, 1)
theta2, m2, v2 = adam_update(theta1, 1.0, m1, v1, 2)
theta3, m3, v3 = adam_update(theta2, 1.0, m2, v2, 3)

# theta must strictly decrease each step (gradient is positive, so theta moves left)
assert theta1 < theta0, "theta must decrease after step 1"
assert theta2 < theta1, "theta must decrease after step 2"
assert theta3 < theta2, "theta must decrease after step 3"

# Total movement over 3 steps should be small (lr=0.001)
assert abs(theta3 - theta0) < 0.1, "|theta_3 - theta_0| should be < 0.1 for lr=0.001"

# After one step with grad=1, m and v must both be positive
assert m1 > 0, "first moment m must be positive after one step with positive gradient"
assert v1 > 0, "second moment v must be positive after one step with positive gradient"

# At t=1 with grad=1: m1 = (1-0.9)*1 = 0.1, v1 = (1-0.999)*1 = 0.001  => v1 < m1
assert v1 < m1, (
    f"v ({v1:.6f}) should be less than m ({m1:.6f}) at t=1 "
    "because v tracks squared gradient with b2=0.999 (much slower decay)"
)

print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def adam_update(theta, grad, m, v, t, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    m = b1 * m + (1 - b1) * grad
    v = b2 * v + (1 - b2) * grad**2
    m_hat = m / (1 - b1**t)
    v_hat = v / (1 - b2**t)
    theta = theta - lr * m_hat / (np.sqrt(v_hat) + eps)
    return theta, m, v
```

</details>

### Exercise 2 — Implement cosine annealing

Cosine annealing smoothly decays the learning rate from $\eta_{\max}$ to
$\eta_{\min}$ over $T$ epochs following a half-cosine curve:

$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\!\left(1 + \cos\!\left(\frac{\pi\, t}{T}\right)\right)$$

At $t=0$: $\cos(0) = 1$, so $\eta_0 = \eta_{\max}$.  
At $t=T$: $\cos(\pi) = -1$, so $\eta_T = \eta_{\min}$.  
At $t=T/2$: $\cos(\pi/2) = 0$, so $\eta_{T/2} = \tfrac{1}{2}(\eta_{\max}+\eta_{\min})$.

Implement `cosine_lr` below for a **scalar** epoch `t` (not an array).

In [ ]:
def cosine_lr(t, T, lr_max, lr_min=0.0):
    """Cosine annealing learning rate at epoch t.

    Args:
        t:      current epoch (0-indexed scalar int or float)
        T:      total number of epochs
        lr_max: maximum (starting) learning rate
        lr_min: minimum (ending) learning rate (default 0.0)

    Returns:
        learning rate at epoch t (scalar float)
    """
    # TODO(you): apply the cosine annealing formula
    # hint: use np.cos and np.pi
    return ...

In [ ]:
# Checks — run me after filling in cosine_lr above
assert abs(cosine_lr(0,   100, 0.1) - 0.1)  < 1e-9, "at t=0 lr should equal lr_max"
assert abs(cosine_lr(100, 100, 0.1) - 0.0)  < 1e-9, "at t=T lr should equal lr_min (0.0)"
assert abs(cosine_lr(50,  100, 0.1) - 0.05) < 1e-9, "at t=T/2 lr should be halfway between max and min"

# Non-zero lr_min
assert abs(cosine_lr(0,   100, 0.1, lr_min=0.01) - 0.1)  < 1e-9
assert abs(cosine_lr(100, 100, 0.1, lr_min=0.01) - 0.01) < 1e-9
assert abs(cosine_lr(50,  100, 0.1, lr_min=0.01) - 0.055) < 1e-9

print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def cosine_lr(t, T, lr_max, lr_min=0.0):
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * t / T))
```

</details>